In [2]:
%pip install anthropic python-dotenv

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [12]:
import json

In [9]:
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"


In [3]:
from dotenv import load_dotenv
from anthropic import Anthropic

# Lee el archivo .env de tu explorador de forma automática
load_dotenv()

True

In [4]:
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)

def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)

def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "temperature": temperature
    }
    if system:
        params["system"] = system
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    
    response = client.messages.create(**params)
    return response.content[0].text

In [10]:
def generate_dataset():
    prompt = """
            Generate an evaluation dataset for a prompt evaluation. The dataset will be used to evaluate prompts that generate Python, JSON, or Regex specifically for AWS-related tasks. Generate an array of JSON objects, each representing task that requires Python, JSON, or a Regex to complete.

            Example output:
            ```json
            [
            {
                "task": "Description of task",
                "format": "json" or "python" or "regex",
                "solution_criteria": "Key criteria for evaluating the solution"
            },
            ...additional
            ]
            ```

            * Focus on tasks that can be solved by writing a single Python function, a single JSON object, or a single regex
            * Focus on tasks that do not require writing much code

            Please generate 3 objects.
            """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```json")


    text = chat(messages, stop_sequences=["```"])

    return json.loads(text)

In [13]:
# Llama a la API (prefill + stop) y parsea la respuesta a lista/dict de Python
dataset = generate_dataset()
print(dataset)

# with open: abre el archivo, escribe y lo cierra solo al salir del bloque (aunque haya error)
# "w" = write (crea o SOBRESCRIBE dataset_generate.json en la carpeta del notebook)
# json.dump convierte la lista/dict de Python a texto JSON en el archivo
# indent=2 = sangría legible (cada nivel con 2 espacios)
with open("dataset_generate.json", "w") as f:
    json.dump(dataset, f, indent=2)

[{'task': "Extract all EC2 instance IDs from AWS CloudTrail logs that match the pattern 'i-' followed by 17 hexadecimal characters", 'format': 'regex', 'solution_criteria': 'Regex should match EC2 instance IDs (i-0123456789abcdef0) and not match partial strings or malformed IDs'}, {'task': "Parse an AWS S3 bucket policy JSON and extract all principals that have 's3:GetObject' permission", 'format': 'python', 'solution_criteria': 'Function should return a list of unique principals (ARNs or principals) that have s3:GetObject permission, handling nested Statement arrays'}, {'task': 'Create a JSON template for an AWS Lambda execution role trust policy that allows the Lambda service to assume the role', 'format': 'json', 'solution_criteria': "JSON should be valid IAM trust policy with correct service principal 'lambda.amazonaws.com' and AssumeRole action"}]


In [ ]:
def run_prompt(test_case):
    """Merges the prompt and test case input, then returns the result"""

    prompt = f"""
            Please solve the following task:

            {test_case["task"]}

            * Response only with Python, JSON, or a plain Regex.
            * Do not add any comments or commentary or explanation.
            """

    messages = []
    add_user_message(messages, prompt)
    add_assistant_message(messages, "```code")
    output = chat(messages, stop_sequences=["```"])
    return output
    

In [75]:
def grade_by_model(test_case, output):
    # Create evaluation prompt
    eval_prompt = f"""
    You are an expert AWS code reviewer. Your task is to evaluate the following AI-Generated solution.
    
    Original Task:
    <task>
    {test_case["task"]}
    </task>

    Solution to Evaluate
    <solution>
    {output}
    </solution>


    Criteria you should use to evaluate the solution:
    <criteria>
    {test_case["solution_criteria"]}
    </criteria>

    
    Provide your evaluation as a structured JSON object with the following fields:
    - "strengths": An array of 1-3 key strengths
    - "weaknesses": An array of 1-3 key areas for improvement  
    - "reasoning": A concise explanation of your assessment
    - "score": A number between 1-10

    Respond with JSON. Keep your response consise and direct
    Example response shape
    {{
        "strengths": string[],
        "weaknesses: string[],
        "reasoning": string,
        "score: number
    }}
    """
    
    messages = []
    add_user_message(messages, eval_prompt)
    add_assistant_message(messages, "```json")
    
    eval_text = chat(messages, stop_sequences=["```"])
    return json.loads(eval_text)

In [76]:
import re
import ast

def validate_json(text):
    try:
        json.loads(text.strip())
        return 10
    except json.JSONDecodeError:
        return 0

def validate_python(text):
    try:
        ast.parse(text.strip())
        return 10
    except SyntaxError:
        return 0

def validate_regex(text):
    try:
        re.compile(text.strip())
        return 10
    except re.error:
        return 0


def grade_syntax(response, test_case):
    format = test_case["format"]
    if format == "json":
        return validate_json(response)
    elif format == "python":
        return validate_python(response)
    else:
        return validate_regex(response)

In [77]:
def run_test_case(test_case):
    """Calls run_prompt, then grades the result"""

    output = run_prompt(test_case)

    # TODO: Grading
    model_grade = grade_by_model(test_case, output)
    model_score = model_grade["score"]
    reasoning = model_grade["reasoning"]



    syntax_score = grade_syntax(output, test_case)

    score = (model_score + syntax_score) / 2

    return {
        "output": output,
        "test_case": test_case,
        "score": score,
        "reasoning": reasoning
    }

In [79]:
from statistics import mean

def run_eval(dataset):
    """Loads the dataset and calls run_test_case with each case"""

    results = []

    for test_case in dataset:
        result = run_test_case(test_case)
        results.append(result)
    
    average_score = mean([result["score"] for result in results])
    print(f"Average score: {average_score}")
    
    return results

In [80]:
with open("dataset_generate.json", "r") as f:
    dataset = json.load(f)

results = run_eval(dataset)

Average score: 4.833333333333333


In [81]:
print(json.dumps(results, indent=2))

[
  {
    "output": "\nimport re\nfrom urllib.parse import urlparse\n\ndef parse_s3_uri(uri):\n    # Parse S3 URI in format: s3://bucket-name/path/to/file\n    match = re.match(r's3://([a-zA-Z0-9._-]+)(/.*)?$', uri)\n    if match:\n        bucket_name = match.group(1)\n        path = match.group(2) or ''\n        return {\n            'bucket': bucket_name,\n            'path': path,\n            'region': None\n        }\n    return None\n\n# Alternative regex-only solution\ndef parse_s3_uri_regex(uri):\n    pattern = r's3://([a-zA-Z0-9._-]+)(/.+)?'\n    match = re.search(pattern, uri)\n    if match:\n        return {\n            'bucket': match.group(1),\n            'path': match.group(2) or '',\n            'region': None\n        }\n    return None\n```\n\n```json\n{\n  \"s3_uri_pattern\": \"s3://([a-zA-Z0-9._-]+)(/.+)?\",\n  \"example_uri\": \"s3://my-bucket/path/to/file\",\n  \"extracted\": {\n    \"bucket\": \"my-bucket\",\n    \"path\": \"/path/to/file\",\n    \"region\": nul